[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-11-file-uploads-forms.ipynb#scrollTo=11a1b2c3)

---
# Day 11 · File Uploads and Form Data
**certified-journeys / fastapi-certified** · Practice Day

> **Goal for today:** Build FastAPI endpoints that accept file uploads and form fields — including content-type validation, streaming large files to disk, and computing checksums — tested end-to-end with `TestClient`.


In [ ]:
%pip install -q fastapi httpx python-multipart


## Step 1 · `UploadFile` vs `bytes` — choosing the right type

FastAPI gives you two ways to receive file data:

| Parameter type | Storage | Max size | Use when |
|---------------|---------|----------|----------|
| `bytes` | Fully in memory | Limited by RAM | Tiny files (< 1 MB thumbnails, config) |
| `UploadFile` | `SpooledTemporaryFile` — RAM up to 1 MB, then disk | Unlimited | Any file you'd serve in production |

`UploadFile` attributes:
- `.filename` — original filename from the browser/client
- `.content_type` — MIME type string (e.g. `"image/jpeg"`)
- `.file` — the underlying `SpooledTemporaryFile` object
- `await .read(size)` — async read
- `await .seek(0)` — rewind for re-reading
- `await .close()` — release resources

> **Rule of thumb:** always use `UploadFile` in production — it streams to disk above 1 MB automatically.


In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.testclient import TestClient
import io

app = FastAPI()

# ── Basic upload endpoint: return filename and size ──────────────────────────
@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    """
    Accept any file, read its contents to measure size,
    then return the original filename and byte count.
    """
    contents = await file.read()     # reads the entire file into memory
    size = len(contents)
    await file.close()               # release the SpooledTemporaryFile
    return {
        "filename": file.filename,
        "content_type": file.content_type,
        "size_bytes": size,
    }

client = TestClient(app)

# ── Test with TestClient: pass files= dict ───────────────────────────────────
def test_basic_upload():
    # files= dict: key = form field name, value = (filename, file-like, content-type)
    fake_image = io.BytesIO(b"FAKEJPEG" * 128)  # 1024 bytes
    response = client.post(
        "/upload",
        files={"file": ("photo.jpg", fake_image, "image/jpeg")}
    )
    assert response.status_code == 200
    data = response.json()
    assert data["filename"] == "photo.jpg"
    assert data["content_type"] == "image/jpeg"
    assert data["size_bytes"] == 1024
    print(f"PASSED: upload → {data}")

test_basic_upload()


### What just happened?
- **`File(...)`** tells FastAPI to expect a multipart/form-data file field — not a JSON body.
- **`await file.read()`** loads the content into a `bytes` object; for large files we'll stream instead (Step 4).
- **`TestClient.post(files=...)`** constructs a multipart request — the tuple `(filename, data, content_type)` matches the browser's `FormData` format.
- `await file.close()` is good hygiene; FastAPI also closes it automatically when the request ends.


## Step 2 · Content-type validation

Browsers send a `Content-Type` header per file part in the multipart body.
FastAPI exposes it as `file.content_type`. We can check it and raise `400 Bad Request` for unsupported types.

**Allowed MIME types for image uploads:**

| MIME type | Extension |
|-----------|----------|
| `image/jpeg` | .jpg / .jpeg |
| `image/png` | .png |

Do not rely only on the extension — a malicious client can rename `exploit.exe` to `photo.jpg`.
Validate the MIME type **and**, for production systems, sniff the first bytes with `python-magic`.


In [ ]:
ALLOWED_TYPES = {"image/jpeg", "image/png"}

@app.post("/upload/image")
async def upload_image(file: UploadFile = File(...)):
    """Accept only JPEG and PNG files."""
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(
            status_code=400,
            detail=f"Unsupported file type '{file.content_type}'. Allowed: {sorted(ALLOWED_TYPES)}"
        )
    contents = await file.read()
    await file.close()
    return {
        "filename": file.filename,
        "content_type": file.content_type,
        "size_bytes": len(contents),
        "accepted": True,
    }

# ── Test: valid PNG ──────────────────────────────────────────────────────────
def test_upload_valid_type():
    png_data = io.BytesIO(b"\x89PNG" + b"\x00" * 100)
    r = client.post("/upload/image", files={"file": ("img.png", png_data, "image/png")})
    assert r.status_code == 200
    assert r.json()["accepted"] is True
    print("PASSED: PNG accepted")

# ── Test: rejected PDF ───────────────────────────────────────────────────────
def test_upload_invalid_type():
    pdf_data = io.BytesIO(b"%PDF-1.4 fake content")
    r = client.post("/upload/image", files={"file": ("report.pdf", pdf_data, "application/pdf")})
    assert r.status_code == 400
    assert "Unsupported file type" in r.json()["detail"]
    print("PASSED: PDF rejected with 400")

test_upload_valid_type()
test_upload_invalid_type()


### What just happened?
- **`file.content_type`** is set by the client in the multipart boundary — it's a string like `"image/png"`.
- We raise `HTTPException(400)` early, before reading file content, to avoid wasting I/O on rejected files.
- The test constructs a fake PNG by starting with the PNG magic bytes `\x89PNG` — illustrates how to simulate real files without loading fixtures.
- **Production hardening:** use `python-magic` (`libmagic` bindings) to check actual file headers, not just the declared content-type.


## Step 3 · Combining form fields and file uploads

HTML forms often send both text fields and files in a single `multipart/form-data` request.
In FastAPI, mix `Form()` parameters with `UploadFile` in the same function signature.

```
Content-Type: multipart/form-data; boundary=----FormBoundaryXYZ

------FormBoundaryXYZ
Content-Disposition: form-data; name="description"

A scenic photo
------FormBoundaryXYZ
Content-Disposition: form-data; name="file"; filename="scene.jpg"
Content-Type: image/jpeg

<binary JPEG data>
------FormBoundaryXYZ--
```

**Note:** You cannot mix `Form()` with a Pydantic model body — use `Form()` fields individually.


In [ ]:
from fastapi import Form

@app.post("/upload/with-description")
async def upload_with_description(
    description: str = Form(...),          # text field from the form
    tags: str = Form(""),                  # optional form field with default
    file: UploadFile = File(...),
):
    """Accept a description + optional tags alongside the file."""
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(400, f"Bad content type: {file.content_type}")

    contents = await file.read()
    await file.close()

    return {
        "filename": file.filename,
        "size_bytes": len(contents),
        "description": description,
        "tags": [t.strip() for t in tags.split(",") if t.strip()],
    }

# ── TestClient: pass data= for form fields, files= for file fields ────────────
def test_upload_with_description():
    jpeg_data = io.BytesIO(b"\xff\xd8\xff" + b"0" * 200)  # JPEG magic bytes
    response = client.post(
        "/upload/with-description",
        data={"description": "Sunset photo", "tags": "travel, nature"},
        files={"file": ("sunset.jpg", jpeg_data, "image/jpeg")},
    )
    assert response.status_code == 200
    body = response.json()
    assert body["description"] == "Sunset photo"
    assert "travel" in body["tags"]
    assert body["filename"] == "sunset.jpg"
    print(f"PASSED: form + file → {body}")

def test_upload_missing_description():
    jpeg_data = io.BytesIO(b"\xff\xd8\xff" + b"0" * 100)
    # Omit 'description' — required Form field — expect 422
    response = client.post(
        "/upload/with-description",
        files={"file": ("x.jpg", jpeg_data, "image/jpeg")},
    )
    assert response.status_code == 422
    print("PASSED: missing Form field → 422")

test_upload_with_description()
test_upload_missing_description()


### What just happened?
- **`data=` and `files=` are separate kwargs** in `TestClient.post()` — `data` sends text form fields, `files` sends the multipart file part.
- FastAPI validates `Form()` fields with the same Pydantic machinery as JSON bodies — missing required fields return 422.
- The tags field uses a comma-separated string approach (instead of repeated fields) to keep the example simple.
- **Magic bytes:** `\xff\xd8\xff` is the JPEG Start of Image (SOI) marker — useful for simulating realistic file payloads in tests.


## Step 4 · Streaming large files — async chunk iteration

For large files, loading the entire content with `await file.read()` can exhaust memory.
Instead, read the `SpooledTemporaryFile` in chunks via the underlying `.file` attribute.

```
Client ──► FastAPI ──► SpooledTemporaryFile
                            │
                     read 64 KB at a time
                            │
                       write to disk
```

**`CHUNK_SIZE = 65536`** (64 KB) is a common default — aligns with OS buffer sizes and keeps memory usage flat regardless of file size.


In [ ]:
import hashlib
import os
import pathlib

CHUNK_SIZE = 65536  # 64 KB

@app.post("/upload/stream")
async def upload_stream(file: UploadFile = File(...)):
    """
    Stream the uploaded file to /tmp in chunks.
    Returns the saved path and SHA-256 checksum.
    Memory usage stays constant regardless of file size.
    """
    dest = pathlib.Path("/tmp") / file.filename
    sha256 = hashlib.sha256()
    total_bytes = 0

    with open(dest, "wb") as out:
        while True:
            # Read up to CHUNK_SIZE bytes from the SpooledTemporaryFile
            chunk = await file.read(CHUNK_SIZE)
            if not chunk:
                break                         # end of file
            out.write(chunk)
            sha256.update(chunk)              # incrementally hash the chunk
            total_bytes += len(chunk)

    await file.close()
    return {
        "saved_path": str(dest),
        "size_bytes": total_bytes,
        "sha256": sha256.hexdigest(),
    }

# ── Test: upload 1 MB of data ────────────────────────────────────────────────
def test_stream_upload():
    # Generate 1 MB of deterministic bytes
    data = bytes(range(256)) * 4096  # 256 * 4096 = 1,048,576 bytes (1 MB)
    expected_sha256 = hashlib.sha256(data).hexdigest()

    response = client.post(
        "/upload/stream",
        files={"file": ("large_file.bin", io.BytesIO(data), "application/octet-stream")}
    )
    assert response.status_code == 200
    body = response.json()

    assert body["size_bytes"] == 1_048_576
    assert body["sha256"] == expected_sha256
    assert body["saved_path"] == "/tmp/large_file.bin"

    # Verify the file was actually written to disk
    assert os.path.exists(body["saved_path"])
    print(f"PASSED: 1 MB streamed, sha256={body['sha256'][:16]}...")

test_stream_upload()


### What just happened?
- **`await file.read(CHUNK_SIZE)`** reads at most 64 KB per iteration — memory usage is bounded.
- **`hashlib.sha256()`** is updated incrementally so we never hold the full file in memory simultaneously with the hash state.
- The file is written to `/tmp` — in production you'd write to an object store (S3, GCS) using `aioboto3` or `google-cloud-storage`.
- The test verifies the on-disk file exists *and* that the checksum matches the original bytes — a complete round-trip assertion.


## Step 5 · Multiple file uploads in one request

FastAPI supports uploading a list of files in a single request using `List[UploadFile]`.
The client sends multiple file parts with the same form field name.

```
POST /upload/batch
Content-Type: multipart/form-data

files: photo1.jpg
files: photo2.png
files: photo3.jpg
```

This is more efficient than making 3 separate HTTP requests and avoids the overhead of 3 round-trips.


In [ ]:
from typing import List

@app.post("/upload/batch")
async def upload_batch(files: List[UploadFile] = File(...)):
    """Accept up to 10 files in one multipart request."""
    if len(files) > 10:
        raise HTTPException(400, "Maximum 10 files per request")

    results = []
    for f in files:
        if f.content_type not in ALLOWED_TYPES:
            results.append({"filename": f.filename, "error": "bad content type"})
            await f.close()
            continue
        contents = await f.read()
        await f.close()
        results.append({
            "filename": f.filename,
            "size_bytes": len(contents),
            "content_type": f.content_type,
        })

    return {"uploaded": len([r for r in results if "error" not in r]), "files": results}

# ── Test: send 3 files at once ───────────────────────────────────────────────
def test_batch_upload():
    files = [
        ("files", ("a.jpg", io.BytesIO(b"\xff\xd8" * 50), "image/jpeg")),
        ("files", ("b.png", io.BytesIO(b"\x89PNG" * 50), "image/png")),
        ("files", ("c.gif", io.BytesIO(b"GIF89a" * 50), "image/gif")),  # will fail
    ]
    # When sending multiple parts with the same name, pass a list of tuples
    response = client.post("/upload/batch", files=files)
    assert response.status_code == 200
    body = response.json()
    # 2 accepted (jpg + png), 1 rejected (gif)
    assert body["uploaded"] == 2
    assert len(body["files"]) == 3
    # The GIF entry should have an error key
    gif_result = next(f for f in body["files"] if f["filename"] == "c.gif")
    assert "error" in gif_result
    print(f"PASSED: batch upload — {body['uploaded']}/3 accepted")

test_batch_upload()


### What just happened?
- **`List[UploadFile]`** tells FastAPI to collect all parts with the same field name into a list.
- `TestClient` accepts a **list of tuples** in `files=` when you need repeated field names.
- Per-file error handling: we `continue` on bad content types rather than failing the whole batch.
- **`await f.close()` in the error branch** is important — prevents file handle leaks even when we skip processing.


## Step 6 · Combined endpoint: form fields, file validation, streaming, checksum

Real upload APIs combine everything: metadata from form fields, type validation, streaming to disk, and returning a checksum so clients can verify integrity.

This is the complete production-ready pattern:

```
POST /upload/complete
  description: str (Form)
  file: UploadFile
  ─────────────────────────────
  1. Validate content_type
  2. Stream to /tmp/<uuid>_<filename>
  3. Compute SHA-256 incrementally
  4. Return {path, size, checksum, description}
```


In [ ]:
import uuid

@app.post("/upload/complete")
async def upload_complete(
    description: str = Form(...),
    file: UploadFile = File(...),
):
    """Full production-ready upload: validate → stream → checksum → respond."""
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(400, f"Rejected: {file.content_type}")

    # Use a UUID prefix to avoid filename collisions in /tmp
    safe_name = f"{uuid.uuid4().hex[:8]}_{file.filename}"
    dest = pathlib.Path("/tmp") / safe_name

    sha256 = hashlib.sha256()
    total = 0

    with open(dest, "wb") as out:
        while True:
            chunk = await file.read(CHUNK_SIZE)
            if not chunk:
                break
            out.write(chunk)
            sha256.update(chunk)
            total += len(chunk)

    await file.close()
    return {
        "description": description,
        "filename": file.filename,
        "saved_path": str(dest),
        "size_bytes": total,
        "sha256": sha256.hexdigest(),
    }

# ── Test the complete endpoint ───────────────────────────────────────────────
def test_upload_complete():
    content = b"\xff\xd8\xff" + b"A" * 500  # ~503 bytes fake JPEG
    expected_hash = hashlib.sha256(content).hexdigest()

    response = client.post(
        "/upload/complete",
        data={"description": "Profile picture"},
        files={"file": ("avatar.jpg", io.BytesIO(content), "image/jpeg")},
    )
    assert response.status_code == 200
    body = response.json()

    assert body["description"] == "Profile picture"
    assert body["filename"] == "avatar.jpg"
    assert body["size_bytes"] == len(content)
    assert body["sha256"] == expected_hash
    assert pathlib.Path(body["saved_path"]).exists()
    print(f"PASSED: complete upload — {body['size_bytes']} bytes, sha256={body['sha256'][:16]}...")

def test_upload_complete_bad_type():
    response = client.post(
        "/upload/complete",
        data={"description": "Should fail"},
        files={"file": ("doc.txt", io.BytesIO(b"hello"), "text/plain")},
    )
    assert response.status_code == 400
    print("PASSED: text/plain rejected with 400")

test_upload_complete()
test_upload_complete_bad_type()


### What just happened?
- **UUID prefix** in the saved filename avoids collisions when multiple users upload files with the same name simultaneously.
- The SHA-256 checksum is computed in the same streaming loop as the write — zero extra passes over the data.
- Two tests: happy path verifies checksum integrity, error test confirms rejection before any file I/O.
- **Production note:** replace `/tmp` with an object store upload via `aioboto3.upload_fileobj()` — the streaming pattern is identical.


## Step 7 · OpenAPI docs for file uploads

FastAPI's auto-generated Swagger UI (`/docs`) renders file upload endpoints correctly.
Understanding what the generated schema looks like helps debug client-side issues.

FastAPI represents `UploadFile` as a `string/binary` schema in the multipart body.
We can inspect the generated OpenAPI spec programmatically.


In [ ]:
import json

# Fetch the OpenAPI schema from the test client
response = client.get("/openapi.json")
schema = response.json()

# Find the /upload/image endpoint schema
upload_image_schema = schema["paths"]["/upload/image"]["post"]

# Show the request body structure
request_body = upload_image_schema.get("requestBody", {})
content_types = list(request_body.get("content", {}).keys())

print("Content types for /upload/image POST:")
for ct in content_types:
    print(f"  {ct}")

# Show the properties of the multipart body
for ct, ct_schema in request_body.get("content", {}).items():
    props = ct_schema.get("schema", {}).get("properties", {})
    print(f"\nProperties under '{ct}':")
    for prop, definition in props.items():
        print(f"  {prop}: {definition}")

print("\nAvailable endpoints:")
for path in schema["paths"]:
    methods = list(schema["paths"][path].keys())
    print(f"  {', '.join(m.upper() for m in methods):6s}  {path}")


### What just happened?
- FastAPI auto-generates an OpenAPI spec available at `/openapi.json` — reachable via `TestClient` just like any endpoint.
- File upload endpoints use `multipart/form-data` content type in the OpenAPI spec, with file fields typed as `string/binary`.
- **Debugging tip:** when a client fails to upload, fetch `/openapi.json` and compare against your client's multipart field names — mismatched names are the most common bug.
- The Swagger UI at `/docs` renders a file picker widget for `UploadFile` fields automatically.


## Step 8 · Edge cases — empty file, filename sanitisation

Production upload endpoints must handle edge cases:

| Edge case | Risk | Mitigation |
|-----------|------|------------|
| Empty file (0 bytes) | Waste storage, corrupt records | Check size after read |
| Path traversal filename | `../../../etc/passwd` written to disk | Sanitise with `pathlib.Path(filename).name` |
| Missing filename | `UploadFile.filename` is `None` | Assign a default UUID name |
| Exceeding size limit | DoS / storage overflow | Check `total` bytes during streaming |


In [ ]:
@app.post("/upload/safe")
async def upload_safe(
    file: UploadFile = File(...),
    max_bytes: int = 5 * 1024 * 1024,  # 5 MB default limit
):
    """Upload with path traversal protection, size limit, and empty-file check."""
    # Sanitise filename: strip directory components
    raw_name = file.filename or "unnamed"
    safe_filename = pathlib.Path(raw_name).name  # strips ../../../
    if not safe_filename:
        safe_filename = uuid.uuid4().hex + ".bin"

    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(400, f"Bad type: {file.content_type}")

    dest = pathlib.Path("/tmp") / safe_filename
    sha256 = hashlib.sha256()
    total = 0

    with open(dest, "wb") as out:
        while True:
            chunk = await file.read(CHUNK_SIZE)
            if not chunk:
                break
            total += len(chunk)
            if total > max_bytes:
                await file.close()
                dest.unlink(missing_ok=True)  # delete partial file
                raise HTTPException(413, f"File exceeds {max_bytes} bytes")
            out.write(chunk)
            sha256.update(chunk)

    await file.close()

    if total == 0:
        dest.unlink(missing_ok=True)
        raise HTTPException(400, "Empty file is not allowed")

    return {"saved": safe_filename, "size_bytes": total, "sha256": sha256.hexdigest()}

# ── Tests for edge cases ─────────────────────────────────────────────────────
def test_empty_file_rejected():
    r = client.post("/upload/safe", files={"file": ("empty.jpg", io.BytesIO(b""), "image/jpeg")})
    assert r.status_code == 400
    assert "Empty" in r.json()["detail"]
    print("PASSED: empty file rejected")

def test_path_traversal_sanitised():
    r = client.post(
        "/upload/safe",
        files={"file": ("../../etc/passwd.jpg", io.BytesIO(b"\xff\xd8" + b"x" * 100), "image/jpeg")}
    )
    assert r.status_code == 200
    # The saved name should be just 'passwd.jpg', not the traversal path
    assert r.json()["saved"] == "passwd.jpg"
    print(f"PASSED: path traversal stripped → saved as '{r.json()['saved']}'")

test_empty_file_rejected()
test_path_traversal_sanitised()


### What just happened?
- **`pathlib.Path(raw_name).name`** extracts only the final component — `../../etc/passwd.jpg` becomes `passwd.jpg`.
- The size check happens **inside the streaming loop** so we stop reading and delete the partial file immediately — no need to buffer the whole thing.
- **`missing_ok=True`** on `unlink` prevents a secondary `FileNotFoundError` if the file wasn't created yet.
- The empty-file check runs **after** the loop — we need to finish streaming to know the total is 0.


In [ ]:
# Challenge: Add a file size limit endpoint
#
# Your task: create POST /upload/limited that:
# 1. Accepts only image/jpeg or image/png
# 2. Enforces a 100 KB (102400 bytes) maximum size
# 3. Returns {filename, size_bytes, sha256} on success
# 4. Returns 413 if the file is too large
# 5. Returns 400 for wrong content type
#
# Then write two tests:
# - test_limited_accepts_small_file: upload 50 KB JPEG → 200
# - test_limited_rejects_large_file: upload 200 KB JPEG → 413
#
# Scaffold:

MAX_UPLOAD_BYTES = 102400  # 100 KB

@app.post("/upload/limited")
async def upload_limited(file: UploadFile = File(...)):
    # YOUR CODE HERE
    # Hint: stream in chunks, track total, raise 413 if exceeded
    pass

def test_limited_accepts_small_file():
    # 50 KB JPEG
    data = b"\xff\xd8\xff" + b"A" * (50 * 1024)
    r = client.post("/upload/limited", files={"file": ("small.jpg", io.BytesIO(data), "image/jpeg")})
    # YOUR ASSERTION HERE
    pass

def test_limited_rejects_large_file():
    # 200 KB JPEG — over the 100 KB limit
    data = b"\xff\xd8\xff" + b"B" * (200 * 1024)
    r = client.post("/upload/limited", files={"file": ("large.jpg", io.BytesIO(data), "image/jpeg")})
    # YOUR ASSERTION HERE
    pass

# Uncomment to run once you've implemented the endpoint:
# test_limited_accepts_small_file()
# test_limited_rejects_large_file()
print("Implement upload_limited() and the two test assertions, then uncomment the calls!")


---
## Day 11 key concepts recap

| Concept | What to remember |
|---|---|
| `UploadFile` vs `bytes` | Always use `UploadFile` in production — streams to disk above 1 MB |
| `file.content_type` | Declared MIME type from multipart header — validate early, before reading |
| `Form()` + `File()` | Mix in the same function; use `data=` + `files=` in TestClient |
| Chunked streaming | `await file.read(CHUNK_SIZE)` in a while loop — flat memory usage |
| Incremental SHA-256 | `sha256.update(chunk)` in the streaming loop — one pass, one hash |
| Path traversal | `pathlib.Path(name).name` strips leading `../` components |
| `List[UploadFile]` | Multiple files in one request; list of tuples in `TestClient.post(files=)` |
| Size limit | Check `total` inside streaming loop, delete partial file on 413 |

> **Tip:** Use `UploadFile` over `bytes` for large files — `UploadFile` streams to a `SpooledTemporaryFile` (disk-backed above 1 MB).

---
## What's next
**Day 12** → WebSockets: real-time bidirectional communication, `ConnectionManager` for multi-client rooms, and testing with `TestClient.websocket_connect()`.

Mark Day 11 complete in your [tracker](../index.html).
